In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
import scipy.io as sio
import copy
from scipy.stats import norm

In [ ]:
# for the font of the plots
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

In [ ]:
# python version of MATLAB smooth function
# from https://stackoverflow.com/questions/40443020/matlabs-smooth-implementation-n-point-moving-average-in-numpy-python

def smooth(a,WSZ):
    # a: NumPy 1-D array containing the data to be smoothed
    # WSZ: smoothing window size needs, which must be odd number,
    # as in the original MATLAB implementation
    
    if len(a)<WSZ and len(a)%2==1:
        WSZ = len(a)
    elif len(a)<WSZ and len(a)%2==0:
        WSZ = len(a)-1
    
    out0 = np.convolve(a,np.ones(WSZ,dtype=int),'valid')/WSZ    
    r = np.arange(1,WSZ-1,2)
    start = np.cumsum(a[:WSZ-1])[::2]/r
    stop = (np.cumsum(a[:-WSZ:-1])[::2]/r)[::-1]
    return np.concatenate((  start , out0, stop  ))

# load data

In [ ]:
# Please contact Professor Tzer Han Tan (tztan@ucsd.edu) for the experimental data

# process the data so that we use the trajectories longer than 10 frames and in the later half of the time
N = 4879 # number of total trajectories
t_half = 400 # the length of the full trajectory is 800. We only use the later half.

# load data
with open('directory_for_expt_data\\Xmat.dat','rb') as f:
    x_arr = np.fromfile(f,np.float64,sep=' ')
x_arr = x_arr.reshape((N,t_half*2))
    
with open('directory_for_expt_data\\Ymat.dat','rb') as f:
    y_arr = np.fromfile(f,np.float64,sep=' ')
y_arr = y_arr.reshape((N,t_half*2))

trajptnum = np.zeros(N) # elements = number of points in ith trajectory
trajlongenough = np.zeros(N) # element = 1 if the number of points > 10

for i in range(N):
    xcopy = copy.copy(x_arr[i,t_half:])
    ptnum = len(xcopy[xcopy!=0]) # length of the chosen trajectory
    trajptnum[i] = ptnum
    if ptnum>10:
        trajlongenough[i]=1 # mark if it is longer than 10 frames

longtrajnum = len(trajlongenough[trajlongenough!=0]) # number of trajectories longer than 10

x_new = np.zeros([longtrajnum,t_half]) # array of the desired trajectories
y_new = np.zeros([longtrajnum,t_half])

idxnew = 0

for i in range(N):
    if trajlongenough[i]==1:
        x_new[idxnew,:] = x_arr[i,t_half:]
        y_new[idxnew,:] = y_arr[i,t_half:]
        idxnew = idxnew + 1
        

# smooth the trajectory and get the deviation

In [ ]:
eqmPosSm=51 # smoothing factor to find mean/eqm position

N = np.size(x_new,axis=0)

xdev=0*x_new
ydev=0*y_new
xeqm=0*x_new
yeqm=0*y_new

for i in range(N):
    xi=x_new[i,:] 
    idx = np.where(xi!=0)
    yi=y_new[i,:]
    
    xdev[i,idx]=xi[idx]-smooth(xi[idx],eqmPosSm)
    ydev[i,idx]=yi[idx]-smooth(yi[idx],eqmPosSm)
    xeqm[i,idx]=smooth(xi[idx],eqmPosSm)
    yeqm[i,idx]=smooth(yi[idx],eqmPosSm)
    

In [ ]:
# non-dimensionalize
pixtomm = 7.8/900 # convert pixel to mm
R = 0.11 # radius of the embryo 0.11 mm

dx = xdev*pixtomm/(2*R) # r = 2R*tilde{r} where tilde{r} is the non-dimensionalized length
dy = ydev*pixtomm/(2*R)
x0 = xeqm*pixtomm/(2*R)
y0 = yeqm*pixtomm/(2*R)


# $\delta v$ from all times

In [ ]:
dt = 10
dv_list_all = []
dvx_list_all = []
dvy_list_all = []

for i in range(N):
    for j in range(t_half-1):
        dxcur = dx[i,j]
        dxnext = dx[i,j+1]
        dycur = dy[i,j]
        dynext = dy[i,j+1]
        
        if dxnext > 0:
            dvx = (dxnext-dxcur)/dt
            dvy = (dynext-dycur)/dt
            
            dv_list_all.append(np.sqrt(dvx**2+dvy**2))
            dvx_list_all.append(dvx)
            dvy_list_all.append(dvy)
            

# get the Gaussian fit

In [ ]:
# need to try different xhalf and yhalf values to get the best Gaussian fit

mu_dvx, std_dvx = norm.fit(dvx_list)
mu_dvy, std_dvy = norm.fit(dvy_list)

xhalf = 0.04
xval = np.linspace(-xhalf, xhalf, 100)
normfit_dvx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_list)*2*xhalf/100

yhalf = 0.042
yval = np.linspace(-yhalf,yhalf,100)
normfit_dvy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_list)*2*yhalf/100

%matplotlib inline

fig = plt.figure(figsize=(10,5),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,2,1)
ax1.hist(np.array(dvx_list),bins=100)
ax1.plot(xval,normfit_dvx,'k')
ax1.text(-0.04,6000,"$\mu$=%s"%('%.4f'%(mu_dvx)),fontsize=15)
ax1.text(-0.04,5500,"$\sigma$=%s"%('%.4f'%(std_dvx)),fontsize=15)
ax1.text(-0.06,6650,'(a)',fontsize=20)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)
ax1.tick_params(labelsize=12)

ax2 = fig.add_subplot(1,2,2)
ax2.hist(np.array(dvy_list),bins=100)
ax2.plot(yval,normfit_dvy,'k')
ax2.text(-0.04,6000,"$\mu$=%s"%('%.4f'%(mu_dvy)),fontsize=15)
ax2.text(-0.04,5500,"$\sigma$=%s"%('%.4f'%(std_dvy)),fontsize=15)
ax2.text(-0.063,6700,'(b)',fontsize=20)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)
ax2.tick_params(labelsize=12)

fig.tight_layout()
plt.show()


# $\delta v$ statistics of a certian t with the most trajectories

In [ ]:
# find the index of t with the most trajectories
maxNumTraj = 0
maxIndex = 0

for i in range(t_half):
    dx_t = dx[:,i]
    numtraj_t = len(dx_t[dx_t!=0])
    if numtraj_t > maxNumTraj:
        maxIndex = i
        maxNumTraj = numtraj_t

In [ ]:
dx_t0 = dx[:,maxIndex]
dy_t0 = dy[:,maxIndex]
dx_t1 = dx[:,maxIndex+1]
dy_t1 = dy[:,maxIndex+1]
x_t0 = x0[:,maxIndex]
x_t1 = x0[:,maxIndex+1]

dt = 10
dv_list = []
dvx_list = []
dvy_list = []

for i in range(N):
    dxcur = dx_t0[i]
    dxnext = dx_t1[i]
    dycur = dy_t0[i]
    dynext = dy_t1[i]
    
    xcur = x_t0[i]
    xnext = x_t1[i]
    
    if xnext > 0 and xcur > 0:
        dvx = (dxnext-dxcur)/dt
        dvy = (dynext-dycur)/dt
            
        dv_list.append(np.sqrt(dvx**2+dvy**2))
        dvx_list.append(dvx)
        dvy_list.append(dvy)
        

# Gaussian fit

In [ ]:
mu_dvx, std_dvx = norm.fit(dvx_list)
mu_dvy, std_dvy = norm.fit(dvy_list)

nBins = 25

xhalf = 0.026
xval = np.linspace(-xhalf, xhalf, 100)
normfit_vx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_list)*2*xhalf/nBins

yhalf = 0.03
yval = np.linspace(-yhalf, yhalf, 100)
normfit_vy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_list)*2*yhalf/nBins

%matplotlib inline

fig = plt.figure(figsize=(8,4),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,2,1)
ax1.hist(dvx_list,bins=nBins)
ax1.plot(xval,normfit_vx,'k')
ax1.text(-0.025,80,"$\mu$=%s"%('%.4f'%(mu_dvx)),fontsize=15)
ax1.text(-0.025,70,"$\sigma$=%s"%('%.4f'%(std_dvx)),fontsize=15)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)

ax2 = fig.add_subplot(1,2,2)
ax2.hist(dvy_list,bins=nBins)
ax2.plot(yval,normfit_vy,'k')
ax2.text(-0.03,80,"$\mu$=%s"%('%.4f'%(mu_dvy)),fontsize=15)
ax2.text(-0.03,70,"$\sigma$=%s"%('%.4f'%(std_dvy)),fontsize=15)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)

fig.tight_layout()
plt.show()


# plot all together

In [ ]:
# plot used in the figure of Supplemental Material

mu_dvx_all, std_dvx_all = norm.fit(dvx_list_all)
mu_dvy_all, std_dvy_all = norm.fit(dvy_list_all)

nBins_all = 100

xhalf_all = 0.04
xval_all = np.linspace(-xhalf_all, xhalf_all, 100)
normfit_dvx_all = norm.pdf(xval_all,mu_dvx_all,std_dvx_all)*len(dvx_list_all)*2*xhalf_all/nBins_all

yhalf_all = 0.042
yval_all = np.linspace(-yhalf_all,yhalf_all,100)
normfit_dvy_all = norm.pdf(yval_all,mu_dvy_all,std_dvy_all)*len(dvy_list_all)*2*yhalf_all/nBins_all

mu_dvx, std_dvx = norm.fit(dvx_list)
mu_dvy, std_dvy = norm.fit(dvy_list)

nBins = 25

xhalf = 0.026
xval = np.linspace(-xhalf, xhalf, 100)
normfit_vx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_list)*2*xhalf/nBins

yhalf = 0.026
yval = np.linspace(-yhalf, yhalf, 100)
normfit_vy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_list)*2*yhalf/nBins

%matplotlib inline

fig = plt.figure(figsize=(20,5),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,4,1)
ax1.hist(dvx_list,bins=nBins)
ax1.plot(xval,normfit_vx,'k')
ax1.text(-0.025,77,"$\mu$=%s"%('%.4f'%(mu_dvx)),fontsize=20)
ax1.text(-0.025,70,"$\sigma$=%s"%('%.4f'%(std_dvx)),fontsize=20)
ax1.text(-0.037,85,'(a)',fontsize=20)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)

ax2 = fig.add_subplot(1,4,2)
ax2.hist(dvy_list,bins=nBins)
ax2.plot(yval,normfit_vy,'k')
ax2.text(-0.03,87,"$\mu$=%s"%('%.4f'%(mu_dvy)),fontsize=20)
ax2.text(-0.03,80,"$\sigma$=%s"%('%.4f'%(std_dvy)),fontsize=20)
ax2.text(-0.044,97,'(b)',fontsize=20)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)

ax3 = fig.add_subplot(1,4,3)
ax3.hist(np.array(dvx_list_all),bins=nBins_all)
ax3.plot(xval_all,normfit_dvx_all,'k')
ax3.text(-0.04,6000,"$\mu$=%s"%('%.4f'%(mu_dvx_all)),fontsize=20)
ax3.text(-0.04,5500,"$\sigma$=%s"%('%.4f'%(std_dvx_all)),fontsize=20)
ax3.text(-0.06,6650,'(c)',fontsize=20)
ax3.set_xlabel('$\delta v_{x}$',fontsize=20)
ax3.set_ylabel('counts',fontsize=20)
ax3.tick_params(labelsize=12)

ax4 = fig.add_subplot(1,4,4)
ax4.hist(np.array(dvy_list_all),bins=nBins_all)
ax4.plot(yval_all,normfit_dvy_all,'k')
ax4.text(-0.04,6000,"$\mu$=%s"%('%.4f'%(mu_dvy_all)),fontsize=20)
ax4.text(-0.04,5500,"$\sigma$=%s"%('%.4f'%(std_dvy_all)),fontsize=20)
ax4.text(-0.063,6700,'(d)',fontsize=20)
ax4.set_xlabel('$\delta v_{y}$',fontsize=20)
ax4.set_ylabel('counts',fontsize=20)
ax4.tick_params(labelsize=12)

fig.tight_layout()
plt.show()
#fig.savefig('hist_dv_idx9_N726_nBin25_Gfit_all_times_v1.pdf',dpi=fig.dpi,bbox_inches='tight')